In [ ]:
import re

In [ ]:
class StateIdToStr(object):

    def __init__(self, module_fname):

        # assumes location of this file w.r.t src; clumsy :/
        fname = os.path.join(os.path.dirname(__file__), '..', 'src', module_fname)
        lines = open(fname, 'r').readlines()
        lines = list(map(str.strip, lines))

        # search forward from start for state register definition
        state_defn_re = re.compile(r"^reg \[.*?\] state.*")
        state_defn_line_num = None
        for i, line in enumerate(lines):
            if state_defn_re.match(line):
                state_defn_line_num = i
                break
        if state_defn_line_num is None:
            raise Exception("couldn't find state register definition")

        # search backwards to find localparam definition
        i = state_defn_line_num - 1
        localparam_line_num = None
        while i > 0:
            if lines[i] == 'localparam':
                localparam_line_num = i
                break
            i -= 1
        if localparam_line_num is None:
            raise Exception("couldn't find localparam defintion")

        # scan between the localparam and state_defn and
        # extract state name to state id mapping
        state_definition_re = re.compile("(.*?)=(.*)[,;]$")
        self.state_id_to_str_dict = {}
        for i in range(localparam_line_num+1, state_defn_line_num):
            line = lines[i]
            # ignore empty lines
            if len(line) == 0:
                continue
            # remve potential trailing comments
            line = re.sub("//.*", '', line).strip()

            # extract name and id
            m = state_definition_re.match(line)
            if not m:
                raise Exception(f"line [{lines[i]}] didn't match expected state definition")
            state_name = m.group(1).strip()
            state_id = int(m.group(2))
            self.state_id_to_str_dict[state_id] = state_name

        if len(self.state_id_to_str_dict) == 0:
            raise Exception("no states extracted?")

    def __getitem__(self, i):
        
        return f"{self.state_id_to_str_dict} ({i})"

In [ ]:


for svfname in ['conv1d.sv', 'dot_product.sv', 'po2_conv1d.sv', 'po2_dot_product.sv', 
                'po2_multiply.sv', 'po2_network.sv', 'qb_network.sv']:

    s2s = StateIdToStr(svfname)
    print(svfname, s2s.state_id_to_str_dict)

In [ ]:
class Foo(object):
    def __init__(self, i):
        self.i = i
    def call(self, j):
        return self.i + j

foo = Foo(3)
foo[4]